<a href="https://colab.research.google.com/github/George7531/Portfolio_Management/blob/main/Group_19296_MScFE_652_Portfolio_Management_Project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Group Work Project # 1**
###**GROUP NUMBER: 19296**
##**Group members:**
- George Immanuel
- Sing Chun Lee
- Duy Vo

#### Import Required Packages

In [ ]:
# This block import all required python packages for running this notebook
import numpy as np                      # numpy
import matplotlib.pyplot as plt         # for plotting
import math                             # math lib
import pandas as pd                     # for storing data in the DataFrame
from pandas.plotting import scatter_matrix
from io import StringIO                 # for using string as io stream
from scipy.integrate import quad        # for numerical integration
from scipy.optimize import brute, fmin  # for searching and minimizing
import yfinance as yf                   # yfinance library
from statsmodels.tsa.stattools import adfuller
import tensorflow as tf                 # tensor flow
!pip install pyts
from pyts.image import GramianAngularField # using GAF
from sklearn.preprocessing import MinMaxScaler
from keras.callbacks import EarlyStopping
from sklearn.preprocessing  import RobustScaler
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from traitlets import validate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.6 MB/s eta 0:00:00


# Data Preperation and Description

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web

# Define parameters
tickers = ['TSLA', 'WMT', 'BAC', 'GS', 'LLY', 'MRK', 'GOOG', 'META', 'AAPL', 'XOM']
BENCHMARK = "^GSPC"
in_sample_start = '2023-09-01'
in_sample_end = '2025-09-30'
out_sample_start = '2025-10-01'
out_sample_end = '2025-12-31'

# 1. Download using yfinance
# Using auto_adjust=False
raw_data = yf.download(
    tickers + [BENCHMARK],
    start='2023-08-25', # For initial return
    end='2025-12-31',
    auto_adjust=False
)

adj_close = raw_data['Close']

# Calculate simple daily percentage returns
daily_returns = adj_close.pct_change().dropna(how='all')

# Drop any trading days where any of the 10 assets have missing data (strict alignment)
daily_returns = daily_returns.dropna()

# In-Sample and Out-of-Sample subsets
returns_in_sample = daily_returns.loc[in_sample_start:in_sample_end]
returns_out_sample = daily_returns.loc[out_sample_start:out_sample_end]

print(f"In-Sample Observations: {len(returns_in_sample)} days")
print(f"Out-of-Sample Observations: {len(returns_out_sample)} days")

# 2. Download Fama-French 5-Factor Daily Data
ff5_daily = web.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start='2023-09-01', end='2025-12-31')[0]

# Convert from percentages to decimals
ff5_daily = ff5_daily / 100.0

# Partition factor data
ff_in_sample = ff5_daily.loc[in_sample_start:in_sample_end]
ff_out_sample = ff5_daily.loc[out_sample_start:out_sample_end]

print("\nFama-French Data Sample:")
print(pd.concat([ff_in_sample.head(2), ff_out_sample.tail(3)]))

[*********************100%***********************]  11 of 11 completed


In-Sample Observations: 521 days
Out-of-Sample Observations: 63 days

Fama-French Data Sample:
            Mkt-RF     SMB     HML     RMW     CMA      RF
Date                                                      
2023-09-01  0.0026  0.0085  0.0058  0.0000 -0.0022  0.0002
2023-09-05 -0.0054 -0.0157 -0.0087 -0.0030 -0.0048  0.0002
2025-12-29 -0.0041 -0.0015  0.0007  0.0032  0.0003  0.0002
2025-12-30 -0.0020 -0.0049  0.0029  0.0036  0.0014  0.0002
2025-12-31 -0.0076  0.0006 -0.0010 -0.0007 -0.0004  0.0002


# Part 2

In [ ]:
########################TO BE REMOVED - REQUIRE STEP 2 OUTPUT FOR PART 2: STEP 4

import scipy.optimize as sco

# Annualize in-sample moments (252 trading days)
rf_annual = 0.045  # 4.50% annualized risk-free rate assumption
mean_returns = returns_in_sample.mean() * 252
cov_matrix = returns_in_sample.cov() * 252

def get_optimal_weights(mean_returns, cov_matrix, max_weight=1.0, rf=0.045):
    num_assets = len(mean_returns)

    # Objective function: Negative Sharpe Ratio
    def min_func_sharpe(weights):
        p_ret = np.dot(weights, mean_returns)
        p_vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(p_ret - rf) / p_vol

    # Constraints: Weights sum to 1.0
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})

    # Bounds: Long-only (0 <= w_i <= max_weight)
    bounds = tuple((0.0, max_weight) for _ in range(num_assets))

    # Initial equal-weight guess
    init_guess = num_assets * [1.0 / num_assets]

    # Optimize using SLSQP
    opt_result = sco.minimize(
        min_func_sharpe,
        init_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    return opt_result.x

# Step 2: Weight-Constrained Optimal Weights (Max weight = 0.18)
w_step2 = get_optimal_weights(mean_returns, cov_matrix, max_weight=0.18, rf=rf_annual)

### Step 4: Factor Correlation Matrix Analysis

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import HuberRegressor
import pandas as pd
import numpy as np

# Factor Correlation Analysis
factors_in_sample = ff_in_sample[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
corr_matrix = factors_in_sample.corr()

print("=== Fama-French 5-Factor In-Sample Correlation Matrix ===")
print(corr_matrix.round(4))

# Compute Daily Excess Portfolio Returns for Step 2 Portfolio (Training Period)
# Convert Fama-French data index to datetime (using to_timestamp if it is a PeriodIndex)
ff_in_sample_dt = ff_in_sample.copy()
if isinstance(ff_in_sample_dt.index, pd.PeriodIndex):
    ff_in_sample_dt.index = ff_in_sample_dt.index.to_timestamp()
else:
    ff_in_sample_dt.index = pd.to_datetime(ff_in_sample_dt.index)

p_ret_in_sample = pd.Series(np.dot(returns_in_sample.values, w_step2), index=pd.to_datetime(returns_in_sample.index))
p_excess_in_sample = p_ret_in_sample - ff_in_sample_dt['RF']

=== Fama-French 5-Factor In-Sample Correlation Matrix ===
        Mkt-RF     SMB     HML     RMW     CMA
Mkt-RF  1.0000  0.2115 -0.2552 -0.3841  0.0249
SMB     0.2115  1.0000  0.3924 -0.4007  0.1535
HML    -0.2552  0.3924  1.0000  0.1861  0.1050
RMW    -0.3841 -0.4007  0.1861  1.0000 -0.1104
CMA     0.0249  0.1535  0.1050 -0.1104  1.0000


### Step 5: OLS and Robust Model Regression Estimation

In [ ]:
factors_dt = ff_in_sample_dt[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]
X_train = sm.add_constant(factors_dt)
y_train = p_excess_in_sample

# 1. OLS Regression with Newey-West (HAC) Standard Errors
ols_model = sm.OLS(y_train, X_train).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

# 2. Huber Robust Regression Estimation
huber = HuberRegressor(epsilon=1.35, max_iter=1000)
huber.fit(factors_dt.values, y_train.values)
huber_coefs = [huber.intercept_] + list(huber.coef_)

# Align indices for OLS Series to match 'Alpha (Intercept)' instead of 'const'
ols_params = ols_model.params.rename(index={'const': 'Alpha (Intercept)'})
ols_bse = ols_model.bse.rename(index={'const': 'Alpha (Intercept)'})
ols_tvalues = ols_model.tvalues.rename(index={'const': 'Alpha (Intercept)'})
ols_pvalues = ols_model.pvalues.rename(index={'const': 'Alpha (Intercept)'})

# Summary Table of Factor Loadings
df_regression = pd.DataFrame({
    'OLS Coef': ols_params,
    'OLS HAC Std Err': ols_bse,
    'OLS t-Stat': ols_tvalues,
    'OLS p-Value': ols_pvalues,
    'Huber Coef': pd.Series(huber_coefs, index=['Alpha (Intercept)', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA'])
})

print("\n=== Fama-French 5-Factor Regression Results (Step 2 Portfolio) ===")
print(df_regression.round(5))

print(f"\nOLS Model R-squared: {ols_model.rsquared:.4f}")
print(f"OLS Model Adjusted R-squared: {ols_model.rsquared_adj:.4f}")


=== Fama-French 5-Factor Regression Results (Step 2 Portfolio) ===
                   OLS Coef  OLS HAC Std Err  OLS t-Stat  OLS p-Value  \
Alpha (Intercept)   0.00038          0.00022     1.76243      0.07800   
Mkt-RF              1.07242          0.02547    42.10910      0.00000   
SMB                -0.15848          0.04694    -3.37606      0.00074   
HML                 0.23816          0.04557     5.22614      0.00000   
RMW                -0.13596          0.06156    -2.20844      0.02721   
CMA                -0.37002          0.07755    -4.77141      0.00000   

                   Huber Coef  
Alpha (Intercept)     0.00034  
Mkt-RF                1.06821  
SMB                  -0.14967  
HML                   0.23623  
RMW                  -0.11565  
CMA                  -0.31793  

OLS Model R-squared: 0.8111
OLS Model Adjusted R-squared: 0.8092


# George's work:
Friends, don't overwrite my report or make any modification to it, just have a look. I'll delete this "George's work" heading later once I complete my contribution.

# part 4: Black Litterman



## install packages

lets install yfinance quietly without showing any warning messages, the '-q' takes care of it.

In [ ]:
!pip -q install yfinance

In [ ]:
## Basic data processing tools
import numpy as np
import pandas as pd

## our beloved yahoo finance
import yfinance as yf

## basic plotting tool
import matplotlib.pyplot as plt

## this is for minimizing negative sharpe ratio or variance of our portfolio
from scipy.optimize import minimize

## for accessing or creating files
from google.colab import files

plan:

1. store all our tickers
2. set s&p 500 index (^GSPC) as our benchmark, define start and end dates, trading days, etc...
3. I want to set the max_weight for the maximum proportion we can invest in any asset in our portfolio to be within 18% as it is one of the constraints mentioned.
4. No short selling is allowed as the weights are supposed to vary between 0 and 0.18 and never become negative.

$$ \sum_{i=1}^{10}w_i=1 $$
$$ 0\leq w_i\leq0.18 $$

In [ ]:
TICKERS = [
    "TSLA", "WMT", "BAC", "GS", "LLY",
    "MRK", "GOOG", "META", "AAPL", "XOM"
]
BENCHMARK = "^GSPC"
START_DATE = "2023-08-31"
END_DATE = "2025-10-02"
TRADING_DAYS = 252
MAX_WEIGHT = 0.18

np.set_printoptions(precision=6, suppress=True)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


download the tickers and s&p 500 historical prices from yahoo finance from start date till end date.

In [ ]:

download = yf.download(
    TICKERS + [BENCHMARK],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False,
    actions=False
)



In [ ]:
download.head(5)

Price       Adj Close                                                        \
Ticker           AAPL       BAC       GOOG         GS        LLY       META   
Date                                                                          
2023-08-31 185.362946 26.681417 136.061295 306.519348 542.899048 293.082947   
2023-09-01 186.931732 26.969915 135.516464 306.229401 545.749634 293.568298   
2023-09-05 187.168549 26.662804 135.427277 302.815491 546.807800 297.302551   
2023-09-06 180.469147 26.420837 134.099854 300.364868 548.531921 296.331848   
2023-09-07 175.190521 26.178869 134.922058 301.141174 561.541016 295.836578   

Price                                                 ...    Volume            \
Ticker           MRK       TSLA       WMT        XOM  ...       BAC      GOOG   
Date                                                  ...                       
2023-08-31 98.779640 258.079987 52.522728 100.754745  ...  37243000  28147900   
2023-09-01 99.559120 245.009995 52.186810 102.866066  ...  35228600  16671600   
2023-09-05 97.447235 256.489990 51.766907 102.875122  ...  46543600  17730200   
2023-09-06 96.522682 251.919998 52.154507 103.763153  ...  45022600  15814300   
2023-09-07 97.836967 251.490005 52.800507 103.255692  ...  35023600  16976000   

Price                                                                 \
Ticker           GS      LLY      META      MRK       TSLA       WMT   
Date                                                                   
2023-08-31  1271100  3043400  17229900  9212000  108861700  19582800   
2023-09-01  1201400  2030800  12842300  4941300  132541600  12553500   
2023-09-05  1846000  2257200  14956000  6379900  129469600  15209400   
2023-09-06  2120800  2424300  15418100  7926100  116959800  14594700   
2023-09-07  1471100  3707100  33748700  8532600  115312900  19120800   

Price                             
Ticker           XOM       ^GSPC  
Date                              
2023-08-31  17479800  3946360000  
2023-09-01  14843500  3246260000  
2023-09-05  18304000  3526250000  
2023-09-06  13325500  3418850000  
2023-09-07  13248500  3763760000  

[5 rows x 66 columns]

Even though we see the adjusted close column is present in our download DataFrame, we want to explicitly check it and raise error just as a convention.

In [ ]:
if "Adj Close" not in download.columns.get_level_values(0):
    raise ValueError("Adjusted Close field was not returned.")

* take a copy of all assets' ajusted close prices.
* set the index to dates
* sort the index in ascending order

In [ ]:
prices_all = download["Adj Close"].copy()
prices_all.index = pd.to_datetime(prices_all.index)
prices_all = prices_all.sort_index()

asset_prices = prices_all[TICKERS]
benchmark_prices = prices_all[BENCHMARK].rename("SP500")



get returns for all assets and benchmark.

In [ ]:
# Daily simple returns
asset_returns_all = asset_prices.pct_change(fill_method=None)
benchmark_returns_all = benchmark_prices.pct_change(fill_method=None)


by default, pct_change method has the tendency to forward fill missing NaN values with previous return, we don't want that to happen so, I explicitly set it to None.

Using only the dates for Black Litterman parameter estimations

In [ ]:

asset_returns = asset_returns_all.loc[
    "2023-09-01":"2025-09-30"
]
benchmark_returns = benchmark_returns_all.loc[
    "2023-09-01":"2025-09-30"
]


All ten stocks and benchmark must be observed on the same dates

In [ ]:
combined = pd.concat(
    [asset_returns, benchmark_returns], axis=1
).dropna()

asset_returns = combined[TICKERS]
benchmark_returns = combined["SP500"]


In [ ]:
len(combined)

521

we have 521 observations in our combined dataframe.

In [ ]:
# our starting is going to be
combined.index.min()

Timestamp('2023-09-01 00:00:00')

In [ ]:
# ending date in the combined dataframe
combined.index.max()

Timestamp('2025-09-30 00:00:00')

In [ ]:
# calculate the missing values(if any)
combined.isna().sum().sum()

np.int64(0)

lets display our beautiful dataFrame

In [ ]:
# lets display our beautiful combined dataFrame
display(combined.head())

,TSLA,WMT,BAC,GS,LLY,MRK,GOOG,META,AAPL,XOM,SP500
Date,,,,,,,,,,,
2023-09-01,-0.050643,-0.006396,0.010813,-0.000946,0.005251,0.007891,-0.004004,0.001656,0.008463,0.020955,0.001799
2023-09-05,0.046855,-0.008046,-0.011387,-0.011148,0.001939,-0.021212,-0.000658,0.012720,0.001267,0.000088,-0.004194
2023-09-06,-0.017817,0.007487,-0.009075,-0.008093,0.003153,-0.009488,-0.009802,-0.003265,-0.035793,0.008632,-0.006972
2023-09-07,-0.001707,0.012386,-0.009158,0.002585,0.023716,0.013616,0.006131,-0.001671,-0.029249,-0.004891,-0.003211
2023-09-08,-0.011889,0.001835,0.008176,0.011057,0.023080,0.010284,0.007342,-0.002611,0.003492,0.014568,0.001427


next up we are going to load the market cap weights, this is the one input we must source from some sec.gov archives so,lets Obtain the S&P 500 index weights of the ten stocks from here. [The site address is here:](https://www.sec.gov/Archives/edgar/data/884394/000119312525297563/d18808dn30d.htm)



In [ ]:
market_weights_df = pd.DataFrame({
    "ticker": [
        "TSLA", "WMT", "BAC", "GS", "LLY",
        "MRK", "GOOG", "META", "AAPL", "XOM"
    ],
    "weight": [
        0.02187476,
        0.00792861,
        0.00616164,
        0.00422510,
        0.01063242,
        0.00367386,
        0.04472378,
        0.02792067,
        0.06623971,
        0.00842514
    ]
})

market_weights_df.to_csv(
    "sp500_weights.csv", index=False
)

files.download("sp500_weights.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

verify the residual portfolio weight.

1. Access the weight column from market_weights_df
2. to get the rest of the companies' weights from s&p 500 apart from these 10 stocks in our portfolio do:  1 - weights_of_ten_stocks_in_our_portfolio.
3. combine the weight of 10 stocks in our portfolio with the weight of rest of the companies in s&p 500.

In [ ]:

w10 = market_weights_df.set_index("ticker")["weight"]

w_rest = 1 - w10.sum()

market_weights = pd.concat([
    w10,
    pd.Series({"REST_SP500": w_rest})
])

display(market_weights.to_frame("market_weight"))



,market_weight
TSLA,0.021875
WMT,0.007929
BAC,0.006162
GS,0.004225
LLY,0.010632
MRK,0.003674
GOOG,0.044724
META,0.027921
AAPL,0.066240
XOM,0.008425


Check whether all eleven weights sum to approximately 1 or 100%

In [ ]:
assert abs(market_weights.sum() - 1) < 1e-8

yes it does!

In [ ]:
print(f"Total market weight: {market_weights.sum():.4%}")
print(f"Residual market weight: {w_rest:.4%}")

Total market weight: 100.0000%
Residual market weight: 79.8194%


We start with ten stocks representing approximately 20.18% of the S&P 500 and construct an eleventh synthetic asset representing the remaining 79.82%

Now we don't have the actual:

* we have to calculate this residual return by multiplying each stock's returns in our portfolio with their corresponding weights and find how much it deviates from the market returns.

$$ r_{\text{rest},t} =
\frac{r_{\text{mkt},t}-\sum_{i=1}^{10}w_i r_{i,t}}
{w_{\text{rest}}} $$

* $ r_{\text{mkt},t } $ is market returns, $ w_i $ is the weight of the i-th asset in our portfolio, $ r_{i,t} $ is the return of our i-th asset, $ w_{\text{rest} } $ is the remaining weight.

* combine the ten stocks return column with the residual return column.

* Then find the maximum error: $$
\text{Maximum Error}
=\max_t |r_{\text{reconstructed},t}-r_{\text{benchmark},t}| $$

* Checks whether the maximum reconstruction error is smaller than \(10^{-10}\).

In [ ]:

# Align stock returns with the benchmark on identical dates.
r10 = asset_returns.copy()
r_mkt = benchmark_returns.copy()

# Synthetic residual return:
# r_SP500 = sum(w_i * r_i) + w_rest * r_rest
r_rest = (
    r_mkt - r10.mul(w10, axis="columns").sum(axis=1)
) / w_rest

r_rest.name = "REST_SP500"

returns_11 = pd.concat([r10, r_rest], axis=1)

market_tickers = TICKERS + ["REST_SP500"]
returns_11 = returns_11[market_tickers].dropna()

# Verify the reconstructed index return.
reconstructed_market = returns_11 @ market_weights
benchmark_aligned = r_mkt.loc[returns_11.index]

max_error = (
    reconstructed_market - benchmark_aligned
).abs().max()

print("Maximum reconstruction error:", max_error)
assert max_error < 1e-10

display(returns_11.head())


Maximum reconstruction error: 3.469446951953614e-18


,TSLA,WMT,BAC,GS,LLY,MRK,GOOG,META,AAPL,XOM,REST_SP500
Date,,,,,,,,,,,
2023-09-01,-0.050643,-0.006396,0.010813,-0.000946,0.005251,0.007891,-0.004004,0.001656,0.008463,0.020955,0.002764
2023-09-05,0.046855,-0.008046,-0.011387,-0.011148,0.001939,-0.021212,-0.000658,0.012720,0.001267,0.000088,-0.006754
2023-09-06,-0.017817,0.007487,-0.009075,-0.008093,0.003153,-0.009488,-0.009802,-0.003265,-0.035793,0.008632,-0.004663
2023-09-07,-0.001707,0.012386,-0.009158,0.002585,0.023716,0.013616,0.006131,-0.001671,-0.029249,-0.004891,-0.002227
2023-09-08,-0.011889,0.001835,0.008176,0.011057,0.023080,0.010284,0.007342,-0.002611,0.003492,0.014568,0.000855
